# Explore waterpoints, settlements, populated places, population density etc

- Aquaya-internal waterpoint data, consolidated
- UNOCHA Populated places (cod-pp)
- HOTOSM Populated places (hotosm-pp)
- HOTOSM Populated place *polygons* (shapes)
- GRID3 settlement extents
  - v1 w/Population
  - v2 w/Extent of urbanization
  - v3 w/simple classification but better segmentation)


*TBD:*
- GRID3 gridded population estimates (density)
- Meta DFG high-res population density
- GHSL extent-of-urbanization latest data (1km grid?)
- Public waterpoint data (mwater, wpdx)

# **Setup**

In [ ]:
import pandas as pd
import geopandas as gpd
import plotly.express as px

In [ ]:
!pip install folium matplotlib mapclassify

In [ ]:
# Sample filepath / load: pd.read_csv("drive/MyDrive/data.csv")
from google.colab import drive
drive.mount('/content/drive')
data_dir = "drive/MyDrive/Colab Notebooks/Data/dd-afkenya/"
data_dir

# **Reference**

Leafleft map tiles:
- OpenStreetMap
- CartoDB positron / dark_matter

# **Config**

In [ ]:
sample_counties = [
    "Uasin Gishu"
]

In [ ]:
# Projected CRS (enabling lat/lon distance comparisons in human units eg: KM)
CRS = "EPSG:21037"

# **Load Data**

## **Admin Boundaries (COD)**

In [ ]:
adm_file = data_dir + "ken_adm_iebc_20191031_shp.zip"
adm_df = gpd.read_file(adm_file, layer="ken_admbnda_adm1_iebc_20191031").to_crs(crs=CRS)
sample_adm_df = adm_df[adm_df["ADM1_EN"].isin(sample_counties)].copy()

adm_df.shape, sample_adm_df.shape

In [ ]:
adm_m = sample_adm_df.explore(style_kwds=dict(color="red", opacity=0.1, fillOpacity=0.1), tiles="CartoDB positron", highlight_kwds=dict(fillOpacity=0.1))

## **Populated Places (COD)**

In [ ]:
# COD Populated Places
codpp_file = data_dir + "KEN_Populated places_2002_DEPHA"

codpp_cols = ['NEWDLAT', 'NEWDLONG', "FULL_NAME", 'DISTRICT', 'REGION', 'LOCATION', 'SUB_LOCATI', 'geometry']
codpp_colmap = {
    "NEWDLAT": "Lat",
    "NEWDLONG": "Lon",
    "FULL_NAME": "Name",
    "REGION": "Province",
    "DISTRICT": "County",
    "LOCATION": "Consituency",
    "SUB_LOCATI": "Ward",
}
codpp_county_map = {
    "E. Marakwet": "Elgeyo-Marakwet",
    "Muranga": "Murang'a",
    "Taita Tavet": "Taita Taveta",
}

codpp_df = gpd.read_file(codpp_file)[codpp_cols].to_crs(crs=CRS)
codpp_df = codpp_df.rename(columns=codpp_colmap)
codpp_df["County"] = codpp_df["County"].replace(codpp_county_map)
sample_codpp_df = codpp_df[codpp_df["County"].isin(sample_counties)].copy()

codpp_df.shape, sample_codpp_df.shape

In [ ]:
pp_m = sample_codpp_df.explore(color="navy", marker_kwds=dict(radius=5))

## **Populated Places (HOTOSM)**

In [ ]:
# HOTOSM Populated Places
hotosm_file = data_dir + "hotosm_ken_populated_places_points_shp.zip"
hotosm_df = gpd.read_file(hotosm_file).to_crs(crs=CRS)

# Drop isolated dwellings
# hotosm_df = hotosm_df[~hotosm_df["place"].isin(["isolated_dwelling"])]

# Sample by intersection w/Adm. boundaries
sample_hotosm_df = hotosm_df[hotosm_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

hotosm_df.shape, sample_hotosm_df.shape

In [ ]:
pp_m = sample_hotosm_df.explore(color="darkorchid", marker_kwds=dict(radius=5))

## **Populated Place *Shapes* (HOTOSM)**

In [ ]:
hotosm_shapes_file = data_dir + "hotosm_ken_populated_places_polygons_shp.zip"
hotshapes_df = gpd.read_file(hotosm_shapes_file).to_crs(crs=CRS)

# Sample by intersection w/Adm Boundaries sample
sample_hotshapes_df = hotshapes_df[hotshapes_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

hotshapes_df.shape, sample_hotshapes_df.shape

In [ ]:
hotshapes_df.head(5)

In [ ]:
# pps_m = sample_hotshapes_df.explore(m=adm_m, color="navy")
# pps_m

## **GRID3 Settlement Extents**

In [ ]:
grid3_v1_disp_cols = ['iso', 'type', 'population', 'pop_un_adj', 'adm1_name', 'adm2_name', 'mgrs_code']
grid3_v3_disp_cols = ['iso3', 'type', 'probability', 'building_count', 'building_area', 'mgrs_code']

### GRID3 SEs V3

In [ ]:
# Settlement Areas (Start w/v3)
se3_file = data_dir + "GRID3_Kenya_Settlement_Extents_Version_3.0/GRID3_KEN_settlement_extents_v3_0.gpkg"
se3_df = gpd.read_file(se3_file).to_crs(crs=CRS)

# Sample via shape intersection (expensive)
sample_se3_df = se3_df[se3_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

se3_df.shape, sample_se3_df.shape

In [ ]:
sample_se3_df.head(2)

In [ ]:
sample_se3_df["type"].value_counts()

In [ ]:
# Custom filters - remove blanket areas (for speed, for now). TODO: Can do this by size of area
# sample_se3_df = sample_se3_df[~sample_se3_df["mgrs_code"].isin(["36MYD5589_1"])].copy()

### GRID3 SEs V1.1

In [ ]:
se1_file = data_dir + "GRID3_Kenya_Settlement_Extents_Version_1.1/GRID3_Kenya_Settlement_Extents_Version_1.1.gdb"
se1_df = gpd.read_file(se1_file).to_crs(crs=CRS)

# Sample via shape intersection (expensive)
sample_se1_df = se1_df[se1_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

se1_df.shape, sample_se1_df.shape

In [ ]:
sample_se1_df.head(2)

### SE Maps

In [ ]:
# Set `se_df` to working copy of settlement extents - whichever version we're using
se_df = se1_df.copy()
sample_se_df = sample_se1_df.copy()

In [ ]:
# se_m = sample_se_df.explore(column="type", cmap=["dodgerblue", "salmon", "violet"], tooltip=grid3_v1_disp_cols, popup=grid3_v1_disp_cols)
# se_m

### SE Population Distributions

In [ ]:
htype, step = "Built-Up Area", 1000
plot_df = sample_se_df[sample_se_df["type"]==htype]

fig = px.histogram(plot_df, x="pop_un_adj", log_x=True,
                   title=f"SE Population Distribution: {htype}",
                   labels={"pop_un_adj": "Population (pop_un_adj)", "type": "Settlement Type"})

fig.update_traces(xbins=dict(start=0.0, end=plot_df["pop_un_adj"].max() + step , size=step))
fig.show()

In [ ]:
htype, step = "Small Settlement Area", 100
plot_df = sample_se_df[sample_se_df["type"]==htype]

fig = px.histogram(plot_df, x="pop_un_adj",
                   title=f"SE Population Distribution: {htype}",
                   labels={"pop_un_adj": "Population (pop_un_adj)", "type": "Settlement Type"})

fig.update_traces(xbins=dict(start=0.0, end=plot_df["pop_un_adj"].max() + step , size=step))
fig.show()

In [ ]:
htype, step = "Hamlet", 100
plot_df = sample_se_df[sample_se_df["type"]==htype]

fig = px.histogram(plot_df, x="pop_un_adj",
                   title=f"SE Population Distribution: {htype}",
                   labels={"pop_un_adj": "Population (pop_un_adj)", "type": "Settlement Type"})

fig.update_traces(xbins=dict(start=0.0, end=plot_df["pop_un_adj"].max() + step , size=step))
fig.show()

## **GRID3 Gridded pop estimates**

In [ ]:
# grid3_grid_file = data_dir + "GRID3_KEN_settlement_grid_v3_0/GRID3_KEN_settlement_grid_v3_0.gpkg"
grid3_grid_file = data_dir + "grid3_ken_settlement_grid_v3_0.zip"
gpd.list_layers(grid3_grid_file)

In [ ]:
grid3_grid_df = gpd.read_file(grid3_grid_file, layer="GRID3_KEN_settlement_grid_v3_0")

In [ ]:
grid3_grid_df.head(1)

## **Aquaya Waterpoints & Labs**

In [ ]:
# Waterpoints
wp_file = data_dir + "AF Kenya - Consolidated Water Systems.xlsx"
wp_df = pd.read_excel(wp_file, sheet_name="Systems")
wp_df = gpd.GeoDataFrame(wp_df, geometry=gpd.points_from_xy(wp_df["Longitude"], wp_df["Latitude"], crs="EPSG:4326")).to_crs(crs=CRS)

# Sample to county
sample_wp_df = wp_df[wp_df["County"].isin(sample_counties)].copy()

wp_df.shape, sample_wp_df.shape

In [ ]:
sample_wp_df.head(2)

In [ ]:
# Labs
labs_df = pd.read_excel(wp_file, sheet_name="Labs")
labs_df = gpd.GeoDataFrame(labs_df, geometry=gpd.points_from_xy(labs_df["Longitude"], labs_df["Latitude"], crs="EPSG:4326")).to_crs(crs=CRS)

sample_labs_df = labs_df[labs_df["County"].isin(sample_counties)].copy()

labs_df.shape, sample_labs_df.shape

In [ ]:
sample_labs_df.head(2)

### Explore Systems

In [ ]:
# wp_m = wp_df.explore(m=adm_m, color="deepskyblue", marker_kwds=dict(radius=10), tiles="CartoDB positron")
# wp_m = labs_df.explore(m=wp_m, color="tomato", marker_kwds=dict(radius=10))
# wp_m

# **All Data Exploration & Analysis**

## **Explore WPs over SEs and PPs**

In [ ]:
# m = sample_adm_df.explore(style_kwds=dict(color="red", opacity=0.05, fillOpacity=0.05), tiles="CartoDB positron", highlight_kwds=dict(fillOpacity=0.05))
# m = sample_se_df.explore(m=m, column="type", cmap=["dodgerblue", "mediumseagreen", "violet"], tooltip=grid3_v1_disp_cols, popup=grid3_v1_disp_cols,
#                           style_kwds_dict=dict(opacity=0.2, fillOpacity=0.2), highlight_kwds=dict(fillOpacity=0.2))
# m = sample_hotosm_df.explore(m=m, color="darkorchid", marker_kwds=dict(radius=5))
# m = sample_wp_df.explore(m=m, color="red", marker_kwds=dict(radius=3))
# m = sample_labs_df.explore(m=m, color="gold", marker_kwds=dict(radius=5))
# m

## **Waterpoint vs SE Stats**

In [ ]:
# Calculate the number and proportion of water points within settlement extents
wp_within_se = gpd.sjoin(sample_wp_df, sample_se_df, how="inner", predicate="within")

num_wp_within_se = len(wp_within_se)
prop_wp_within_se = num_wp_within_se / len(sample_wp_df)

print(f"WPs within settlement extents: {num_wp_within_se} (total {len(sample_wp_df)}; {prop_wp_within_se:.2f})")

In [ ]:
# Calculate the number and proportion of water points within 500 meters of settlement extents
# First, create a buffer around the settlement extents
sample_se_df_buffer = sample_se_df.copy()
sample_se_df_buffer['geometry'] = sample_se_df_buffer.geometry.buffer(500)

# Perform a spatial join between water points and the buffered settlement extents
# To avoid counting water points multiple times if they are within the buffer of
# multiple settlement extents, drop duplicates based on the water point index.
wp_within_buffer = gpd.sjoin(sample_wp_df, sample_se_df_buffer, how="inner", predicate="within")
wp_within_buffer_unique = wp_within_buffer[~wp_within_buffer.index.duplicated(keep='first')].copy()


num_wp_within_buffer = len(wp_within_buffer_unique)
prop_wp_within_buffer = num_wp_within_buffer / len(sample_wp_df)

print(f"WPs within 500 meters of settlement extents: {num_wp_within_buffer} (total {len(sample_wp_df)}; {prop_wp_within_buffer:.2f})")

In [ ]:
# Break down the number and proportion of water points within settlement extents by settlement type
wp_within_se_by_type = wp_within_se.groupby("type").size().reset_index(name="count")
wp_within_se_by_type["proportion"] = wp_within_se_by_type["count"] / num_wp_within_se

# Count the number of unique settlement extents water points fall into for each settlement type
unique_se_with_wp_by_type = wp_within_se.groupby("type")["mgrs_code"].nunique().reset_index(name="unique_settlement_extents_count")

# Merge the two dataframes
combined_stats_by_type = pd.merge(wp_within_se_by_type, unique_se_with_wp_by_type, on="type")

print("\nCombined statistics by settlement type:")
display(combined_stats_by_type)

## **SE w/Waterpoint Stats**

In [ ]:
# Identify the unique settlement extents that contain water points
unique_se_with_wp = wp_within_se[~wp_within_se["mgrs_code"].duplicated(keep='first')].copy()
unique_se_with_wp.shape

In [ ]:
# Calculate the number and proportion of settlement extents with a waterpoint within them
num_se_with_wp = len(unique_se_with_wp)
prop_se_with_wp = num_se_with_wp / len(sample_se_df)

print(f"\nSettlement extents with at least one waterpoint: {num_se_with_wp} (total {len(sample_se_df)}; {prop_se_with_wp:.2f})")

# Break down the number and proportion of settlement extents with a waterpoint by settlement type
se_with_wp_by_type = unique_se_with_wp.groupby("type").size().reset_index(name="count")
se_with_wp_by_type["proportion"] = se_with_wp_by_type["count"] / num_se_with_wp

# Calculate average size and population by settlement type
avg_stats_by_type = unique_se_with_wp.groupby("type")[["Shape_Area", "pop_un_adj"]].mean().reset_index()

# Merge the two dataframes
combined_se_stats = pd.merge(se_with_wp_by_type, avg_stats_by_type, on="type")

print("\nCombined statistics for settlement extents with waterpoints by type:")
display(combined_se_stats)

In [ ]:
# Calculate the average size (Shape_Area) and population (pop_un_adj) of the shapes that have waterpoints within them
average_size = unique_se_with_wp["Shape_Area"].mean()
average_population = unique_se_with_wp["pop_un_adj"].mean()

print(f"\nAverage size (Shape_Area) of settlement extents with waterpoints: {average_size:.2f}")
print(f"Average population (pop_un_adj) of settlement extents with waterpoints: {average_population:.2f}")

In [ ]:
# Hists of SE population for SEs w/WPs
fig1 = px.histogram(unique_se_with_wp[unique_se_with_wp["type"]=="Built-Up Area"], x="pop_un_adj", log_x=True, width=800, title="BUAs w/WP")
fig2 = px.histogram(unique_se_with_wp[unique_se_with_wp["type"]=="Small Settlement Area"], x="pop_un_adj", width=800, title="SSas w/WP")
fig3 = px.histogram(unique_se_with_wp[unique_se_with_wp["type"]=="Hamlet"], x="pop_un_adj", width=800, title="Hamlets w/WP")

fig1.update_traces(xbins=dict(start=0.0, end=8000000 , size=1000))
fig2.update_traces(xbins=dict(start=0.0, end=6000, size=100))
fig3.update_traces(xbins=dict(start=0.0, end=300, size=50))
fig1.show(), fig2.show(), fig3.show()